# Can the mtanh baseline fit be trusted at the pedestal?

No scaling anywhere in this notebook. It fits, and measures the fit.

**Why it exists.** The sparse-grid axes are scale factors on mtanh fit
parameters, so a bad fit does not produce a bad point — it redefines what every
axis value means, for the whole scan, silently. A first pass flagged
`rms_relative` at 2.7–3.0% on 132588 and 11.8% on 129038's `ne`, against 0.3–0.6%
on 132543.

**But that number is the wrong test**, and this notebook exists to replace it.
`rms_relative` is computed over the *whole* profile, core to separatrix. A fit
can score badly because it misses the core — which the scan never touches — and
still be excellent where the pedestal axes act. The reverse is also possible: a
respectable global number hiding a bad pedestal.

So: error as a function of radius, and a total taken only over the pedestal
window. That is the number that licenses the scale factors.

**Two parameterizations, both fitted.**

- `fit_mtanh` — Bruncrona et al. 2025 Eq. 4, a *pedestal-only* form. What
  `apply_mtanh_ped` uses, so its pedestal error is the one that governs the
  current axes.
- `fit_mtanh_full` — Stefanikova 2016, axis-to-SOL in one formula. What
  `apply_mtanh_full` would use. Included because if the pedestal-only form is
  the thing failing, the global form is the ready alternative.

In [1]:
import os, json
import numpy as np
import matplotlib.pyplot as plt

from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics
from TPED.projects.discharge_tools.src.transforms.mtanh_transforms import (
    fit_mtanh, fit_mtanh_full)

DISCHARGE_ROOT = r"C:/Users/joesc/git/ST_research/NSTXU_discharges"

# 129038's directory holds five pfiles, so auto-discovery refuses to guess and
# needs an explicit one. Everything else auto-discovers.
DISCHARGES = {
    129015: {},
    129038: {"pfile": "p129038.00400"},
    132543: {},
    132588: {},
}

# Profiles the scan axes act on, plus the ones that follow through
# quasineutrality — a fit that is fine for ne and wrong for ni still matters,
# because CHEASE sees both.
VARS = ["Te", "Ti", "ne", "ni"]

# The window the verdict is taken over. Inside it the fit governs the axes;
# outside, error is real but does not touch the scan. 0.95 rather than 1.0
# deliberately: the last few points are separatrix/SOL, where the pfile itself
# is least trustworthy and where nothing is scanned.
PED_WINDOW = (0.60, 0.95)

# Weighting used for the pedestal fit throughout, matching the campaign.
FIT_KWARGS = dict(pedestal_weight=8.0)

PED_LO, PED_HI = PED_WINDOW
print(f"verdict window: rho_tor {PED_LO} to {PED_HI}")

verdict window: rho_tor 0.6 to 0.95


## Fit every profile of every discharge, both parameterizations

`fit_mtanh_full` is allowed to fail without taking the notebook with it — the
Stefanikova form has more parameters and does not always converge on an ST
profile. A failure there is a result, not an error.

In [2]:
def load(shot):
    d = os.path.join(DISCHARGE_ROOT, str(shot))
    kw = {"input_dir": d}
    name = DISCHARGES[shot].get("pfile")
    if name:
        kw["pfile"] = os.path.join(d, name)
    return DischargeData(**kw)


results = {}
for shot in DISCHARGES:
    phys = DischargePhysics(load(shot))
    x = np.asarray(phys.rhot.values)
    entry = {"x": x, "vars": {}}
    for var in VARS:
        if var not in phys.ds:
            continue
        y = np.asarray(getattr(phys, var).values)
        # Normalize the error by the profile RANGE, not by y itself: a relative
        # error against a value that goes to ~0 at the separatrix explodes there
        # and would swamp the pedestal, which is the region under test.
        scale = float(np.max(y) - np.min(y)) or 1.0
        rec = {"y": y, "scale": scale}
        for label, fn in (("ped", fit_mtanh), ("full", fit_mtanh_full)):
            try:
                profile, meta = fn(phys.ds, var, **FIT_KWARGS)
                yhat = np.asarray(profile(x), dtype=float)
                rec[label] = {"yhat": yhat,
                              "err": (yhat - y) / scale,
                              "rms_global": meta["rms_relative"],
                              "params": meta["fit_params"]}
            except Exception as exc:
                rec[label] = {"error": f"{type(exc).__name__}: {exc}"}
                print(f"  {shot} {var} {label}: FAILED {rec[label]['error']}")
        entry["vars"][var] = rec
    results[shot] = entry
    print(f"{shot}: fitted {list(entry['vars'])}")

ValueError: Cannot read directory: C:/Users/joesc/git/ST_research/NSTXU_discharges/129015

## Error against radius — core to separatrix

One panel per discharge, every profile on it, both parameterizations
(pedestal-only solid, full dashed). The shaded band is the verdict window.

What to look for: error that is large in the core and small inside the band is
*fine* — the scan does not touch the core. Error that rises inside the band is
what disqualifies a fit.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.0 * len(results), 4.2),
                         sharey=True, squeeze=False)
colors = {"Te": "tab:red", "Ti": "tab:orange", "ne": "tab:blue", "ni": "tab:cyan"}

for ax, (shot, entry) in zip(axes[0], sorted(results.items())):
    x = entry["x"]
    for var, rec in entry["vars"].items():
        for label, style in (("ped", "-"), ("full", "--")):
            f = rec.get(label, {})
            if "err" not in f:
                continue
            ax.plot(x, 100 * f["err"], style, lw=1.1, color=colors.get(var),
                    label=f"{var} {label}", alpha=0.9)
    ax.axvspan(PED_LO, PED_HI, color="k", alpha=0.07)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xlim(0, 1.0); ax.set_ylim(-15, 15)
    ax.set_xlabel("rho_tor"); ax.set_title(str(shot))
axes[0][0].set_ylabel("(fit - data) / range   [%]")
axes[0][-1].legend(fontsize=6, ncol=2)
fig.suptitle("mtanh fit error vs radius — solid: pedestal form, dashed: full form"
             f"   (shaded = verdict window {PED_LO}-{PED_HI})")
plt.tight_layout(); plt.show()

## The verdict — error inside the pedestal window only

`rms_ped` and `max_ped` are taken over the window alone. `rms_global` is the
whole-profile number for comparison: where the two diverge, the global number
was measuring something the scan does not care about.

In [ ]:
def window_stats(x, err):
    m = (x >= PED_LO) & (x <= PED_HI)
    e = err[m]
    return (float(np.sqrt(np.mean(e ** 2))), float(np.max(np.abs(e))))


TRUST = 0.01     # 1% of profile range, rms, inside the window

rows = []
for shot, entry in sorted(results.items()):
    x = entry["x"]
    for var, rec in entry["vars"].items():
        for label in ("ped", "full"):
            f = rec.get(label, {})
            if "err" not in f:
                rows.append({"shot": shot, "var": var, "form": label,
                             "rms_ped": float("nan"), "max_ped": float("nan"),
                             "rms_global": float("nan"), "ok": False,
                             "note": f.get("error", "")[:40]})
                continue
            rms, mx = window_stats(x, f["err"])
            rows.append({"shot": shot, "var": var, "form": label,
                         "rms_ped": rms, "max_ped": mx,
                         "rms_global": f["rms_global"],
                         "ok": rms <= TRUST, "note": ""})

hdr = (f"{'shot':>7} {'var':<4} {'form':<5} {'rms_ped':>9} {'max_ped':>9} "
       f"{'rms_global':>11} {'verdict':>9}")
print(hdr); print("-" * len(hdr))
for r in rows:
    v = "TRUST" if r["ok"] else "reject"
    rp = f"{r['rms_ped']*100:>8.2f}%" if r["rms_ped"] == r["rms_ped"] else f"{'--':>9}"
    mp = f"{r['max_ped']*100:>8.2f}%" if r["max_ped"] == r["max_ped"] else f"{'--':>9}"
    rg = f"{r['rms_global']*100:>10.2f}%" if r["rms_global"] == r["rms_global"] else f"{'--':>11}"
    print(f"{r['shot']:>7} {r['var']:<4} {r['form']:<5} {rp} {mp} {rg} {v:>9}"
          + (f"   {r['note']}" if r["note"] else ""))
print(f"\nTRUST threshold: rms inside {PED_LO}-{PED_HI} at or below "
      f"{TRUST:.0%} of profile range")

### Per-discharge summary for the axes actually in use

The scan axes are `Te_ped_scale` and `ne_ped_scale`, both driving
`apply_mtanh_ped` — so the **pedestal form's** `Te` and `ne` rows are what
license them. `Ti`/`ni` follow through quasineutrality and the closure, so a bad
fit there reaches CHEASE even though no axis names it.

In [ ]:
print(f"{'shot':>7}  {'Te ped':>8} {'ne ped':>8}   axes usable?")
print("-" * 48)
verdict = {}
for shot, entry in sorted(results.items()):
    x = entry["x"]
    vals = {}
    for var in ("Te", "ne"):
        f = entry["vars"].get(var, {}).get("ped", {})
        vals[var] = window_stats(x, f["err"])[0] if "err" in f else float("nan")
    ok = all(v == v and v <= TRUST for v in vals.values())
    verdict[shot] = {"Te_rms_ped": vals["Te"], "ne_rms_ped": vals["ne"], "ok": ok}
    print(f"{shot:>7}  {vals['Te']*100:>7.2f}% {vals['ne']*100:>7.2f}%   "
          f"{'yes' if ok else 'NO — refit before scaling'}")

out = "mtanh_fit_quality.json"
with open(out, "w") as f:
    json.dump({"ped_window": [PED_LO, PED_HI], "trust_threshold": TRUST,
               "rows": [{k: v for k, v in r.items()} for r in rows],
               "verdict": verdict}, f, indent=1, default=str)
print(f"\nwritten: {out}")

## If a fit is rejected — what to try

In order of effort. Re-run the cells above after each; the verdict table is the
arbiter.

1. **Raise `pedestal_weight`.** Currently 8. It upweights points inside
   `ped_threshold`–`edge_threshold`, buying pedestal accuracy at the core's
   expense — which is the trade this window says we want.
2. **Narrow `ped_threshold`.** Default 0.85. A fit whose `psi_mid` lands exactly
   on 0.85 has hit that bound rather than found the pedestal; moving the
   boundary lets it search where the pedestal actually is.
3. **Supply explicit `p0`.** Eight parameters — `[y_sep, a_y0, delta_ped,
   psi_mid, a_y1, psi_ped, alpha_1, alpha_2]`. Seed `psi_mid` and `delta_ped`
   from the profile by eye; those two are what the auto-guess gets wrong.
4. **Switch that discharge to `apply_mtanh_full`.** If the full form's pedestal
   error beats the pedestal form's in the table above, the axis should be
   defined on it instead — at the cost of a different, larger parameter set.

In [ ]:
for shot, entry in sorted(results.items()):
    x = entry["x"]
    for var in ("Te", "ne"):
        rec = entry["vars"].get(var, {})
        p, fl = rec.get("ped", {}), rec.get("full", {})
        if "err" not in p:
            continue
        rp = window_stats(x, p["err"])[0]
        if rp <= TRUST:
            continue
        params = p["params"]
        print(f"{shot} {var}: rms_ped {rp*100:.2f}%  "
              f"psi_mid {params.get('psi_mid'):.4f}  "
              f"delta_ped {params.get('delta_ped'):.4f}")
        if abs(params.get("psi_mid", 0) - 0.85) < 1e-6:
            print("    psi_mid is ON the ped_threshold bound — try step 2 first")
        if "err" in fl:
            rf = window_stats(x, fl["err"])[0]
            better = "BETTER" if rf < rp else "no better"
            print(f"    full form rms_ped {rf*100:.2f}% — {better}")